# Inferencia con los modelos entrenados
Este notebook carga los modelos originales entrenados con W01 y realiza inferencia sobre PC/combined_features.csv.
No se utilizan modelos de ablacion.

In [11]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

## Cargar datos y modelos

In [12]:
base_dir = Path.cwd()
data_path = base_dir / 'PC' / 'combined_features.csv'
if not data_path.exists():
    raise FileNotFoundError(f'No se encontro el archivo: {data_path}')

features = [
    'ECG_mean', 'ECG_std', 'ECG_range', 'ECG_energy',
    'ECG_samp_ent', 'ECG_missing_peaks', 'SDNN', 'RMSSD',
    'pNN50', 'HR_mean', 'HR_max', 'HR_min'
]

df = pd.read_csv(data_path)
missing_columns = [column for column in features + ['FatigueIndex'] if column not in df.columns]
if missing_columns:
    raise ValueError(f'Faltan columnas requeridas: {missing_columns}')

X = df[features].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.mean())

linear_model = joblib.load(base_dir / 'linear_regression_model.pkl')
svm_model = joblib.load(base_dir / 'svm_regression_model.pkl')
svm_scaler = joblib.load(base_dir / 'svm_regression_scaler.pkl')
xgb_regression_model = joblib.load(base_dir / 'xgboost_regression_model.pkl')
xgb_classifier_model = joblib.load(base_dir / 'xgboost_classifier_model.pkl')
classifier_thresholds = joblib.load(base_dir / 'xgboost_classifier_thresholds.pkl')

print(f'Datos cargados: {len(df)} ventanas')
print(f'Caracteristicas utilizadas: {len(features)}')

Datos cargados: 305 ventanas
Caracteristicas utilizadas: 12


## Predicciones de los modelos de regresion

In [13]:
results = df[['UNIX Timestamp', 'DateTime']].copy()
results['FatigueIndex_real'] = df['FatigueIndex']
results['FatigueIndex_linear'] = linear_model.predict(X)

X_svm_scaled = svm_scaler.transform(X)
results['FatigueIndex_svm'] = svm_model.predict(X_svm_scaled)
results['FatigueIndex_xgboost'] = xgb_regression_model.predict(X)

results.head()

,UNIX Timestamp,DateTime,FatigueIndex_real,FatigueIndex_linear,FatigueIndex_svm,FatigueIndex_xgboost
0,1787922065267126924,2026-08-28T14:01:05.267127,1.248597,1.248597,0.151669,1.345281
1,1787922095267568940,2026-08-28T14:01:35.267569,-4.209441,-4.209441,-3.349090,-1.404372
2,1787922125274596503,2026-08-28T14:02:05.274597,2.233302,2.233302,1.478035,2.186211
3,1787922155281166332,2026-08-28T14:02:35.281166,-4.103123,-4.103123,-3.719027,-2.054645
4,1787922185287797157,2026-08-28T14:03:05.287797,2.683160,2.683160,2.123301,2.538092


## Prediccion del clasificador XGBoost

In [14]:
class_predictions = xgb_classifier_model.predict(X).astype(int)
class_names = {0: 'Bajo', 1: 'Medio', 2: 'Alto'}
results['Fatigue_class_number'] = class_predictions
results['Fatigue_class'] = pd.Series(class_predictions, index=results.index).map(class_names)

print(f"Umbral P33: {classifier_thresholds['P33']}")
print(f"Umbral P66: {classifier_thresholds['P66']}")
results.head()

Umbral P33: -5.021884072987618
Umbral P66: -4.517375064139081


,UNIX Timestamp,DateTime,FatigueIndex_real,FatigueIndex_linear,FatigueIndex_svm,FatigueIndex_xgboost,Fatigue_class_number,Fatigue_class
0,1787922065267126924,2026-08-28T14:01:05.267127,1.248597,1.248597,0.151669,1.345281,2,Alto
1,1787922095267568940,2026-08-28T14:01:35.267569,-4.209441,-4.209441,-3.349090,-1.404372,2,Alto
2,1787922125274596503,2026-08-28T14:02:05.274597,2.233302,2.233302,1.478035,2.186211,2,Alto
3,1787922155281166332,2026-08-28T14:02:35.281166,-4.103123,-4.103123,-3.719027,-2.054645,2,Alto
4,1787922185287797157,2026-08-28T14:03:05.287797,2.683160,2.683160,2.123301,2.538092,2,Alto


## Comparacion con FatigueIndex generado
Estas metricas son descriptivas porque FatigueIndex fue construido a partir de las mismas caracteristicas de entrada.

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

metrics_rows = []
target_values = results['FatigueIndex_real']
target_mean_absolute = target_values.abs().mean()

for model_name in ['linear', 'svm', 'xgboost']:
    prediction_column = f'FatigueIndex_{model_name}'
    prediction = results[prediction_column]
    mse = mean_squared_error(target_values, prediction)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(target_values, prediction)
    r2 = r2_score(target_values, prediction)
    mae_relative = (mae / target_mean_absolute) * 100
    metrics_rows.append({
        'Modelo': model_name.capitalize(),
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAE relativo (%)': mae_relative,
        'R² (%)': r2 * 100
    })

metrics_table = pd.DataFrame(metrics_rows)
metrics_table[['MSE', 'RMSE', 'MAE']] = metrics_table[['MSE', 'RMSE', 'MAE']].round(4)
metrics_table[['MAE relativo (%)', 'R² (%)']] = metrics_table[['MAE relativo (%)', 'R² (%)']].round(2)
metrics_table

,Modelo,MSE,RMSE,MAE,MAE relativo (%),R² (%)
0,Linear,0.0000,0.0000,0.0000,0.00,100.00
1,Svm,2.0418,1.4289,0.4518,12.33,76.44
2,Xgboost,4.9989,2.2358,1.7301,47.22,42.32


## Guardar resultados

In [16]:
output_path = base_dir / 'inference_results.csv'
results.to_csv(output_path, index=False)
print(f'Resultados guardados en: {output_path}')

Resultados guardados en: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\2025-08-11 ECG\2025-08-11\WCarlos\inference_results.csv
